# Missing Values and Data Types

Real-world data is messy. Missing values and incorrect data types are the most common data quality issues you'll encounter. Handling them properly is essential before any analysis or modeling.

## Learning Objectives

- Detect missing values with `isna()`, `isnull()`, and summary methods
- Handle missing values with `dropna()` and `fillna()`
- Convert data types with `astype()`, `to_numeric()`, `to_datetime()`
- Use `pd.Categorical` for memory-efficient categorical data
- Work with string data using the `.str` accessor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load data
df = pd.read_csv('../../data/titanic.csv')
print(f"Shape: {df.shape}")
df.head()

## Detecting Missing Values

In [ ]:
# isna() and isnull() are identical
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
# As percentages
print("\nMissing value percentages:")
missing_pct = (df.isna().sum() / len(df) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

In [ ]:
# notna() - the opposite
print(f"\nRows with non-null Age: {df['Age'].notna().sum()}")
print(f"Rows with null Age: {df['Age'].isna().sum()}")

In [ ]:
# Any row with any missing value
rows_with_missing = df[df.isna().any(axis=1)]
print(f"Rows with at least one missing value: {len(rows_with_missing)}")

## Visualizing Missing Values

In [ ]:
# Heatmap of missing values
fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Value Heatmap\n(Yellow = Missing)')
ax.set_xlabel('Column')
ax.set_ylabel('Row')

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart of missing values
fig, ax = plt.subplots(figsize=(10, 5))

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

bars = ax.bar(missing.index, missing.values, color='steelblue')
ax.set_ylabel('Number of Missing Values')
ax.set_title('Missing Values by Column')

# Add percentage labels
for bar, val in zip(bars, missing.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            f'{pct:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## Handling Missing Values: dropna()

In [ ]:
# dropna() removes rows with missing values
df_clean = df.dropna()
print(f"Original rows: {len(df)}")
print(f"After dropna(): {len(df_clean)}")
print(f"Dropped: {len(df) - len(df_clean)} rows")

In [ ]:
# dropna() parameters

# how='any' (default) - drop if ANY column is null
# how='all' - drop only if ALL columns are null
df_all = df.dropna(how='all')
print(f"With how='all': {len(df_all)} rows (no all-null rows)")

In [ ]:
# subset: only consider specific columns
df_age_not_null = df.dropna(subset=['Age'])
print(f"Rows with non-null Age: {len(df_age_not_null)}")

df_age_embarked = df.dropna(subset=['Age', 'Embarked'])
print(f"Rows with non-null Age AND Embarked: {len(df_age_embarked)}")

In [ ]:
# Drop columns instead of rows
df_no_cabin = df.dropna(axis=1)  # axis=1 means columns
print(f"Original columns: {df.columns.tolist()}")
print(f"Columns after dropna(axis=1): {df_no_cabin.columns.tolist()}")

## Handling Missing Values: fillna()

In [ ]:
# Fill with a scalar value
df_filled = df.copy()
df_filled['Age'] = df_filled['Age'].fillna(0)
print(f"Age missing after fillna(0): {df_filled['Age'].isna().sum()}")

In [ ]:
# Fill with mean/median/mode
df_filled = df.copy()

# Mean imputation
mean_age = df['Age'].mean()
df_filled['Age'] = df_filled['Age'].fillna(mean_age)
print(f"Filled Age with mean: {mean_age:.1f}")

# Mode imputation (for categorical)
mode_embarked = df['Embarked'].mode()[0]
df_filled['Embarked'] = df_filled['Embarked'].fillna(mode_embarked)
print(f"Filled Embarked with mode: {mode_embarked}")

In [ ]:
# Forward fill and backward fill (useful for time series)
# ffill: propagate last valid value forward
# bfill: propagate next valid value backward

s = pd.Series([1, np.nan, np.nan, 4, np.nan, 6])
print(f"Original: {s.tolist()}")
print(f"ffill: {s.ffill().tolist()}")
print(f"bfill: {s.bfill().tolist()}")

## Data Type Conversion

In [ ]:
print("Current dtypes:")
print(df.dtypes)

In [ ]:
# astype() - basic conversion
df_copy = df.copy()

# Convert Pclass to string
df_copy['Pclass_str'] = df_copy['Pclass'].astype(str)
print(f"Pclass dtype: {df_copy['Pclass'].dtype}")
print(f"Pclass_str dtype: {df_copy['Pclass_str'].dtype}")

In [ ]:
# pd.to_numeric() - safer for messy data
# errors='coerce' converts invalid values to NaN instead of raising error

messy_data = pd.Series(['1', '2', '3', 'four', '5', 'N/A'])
print(f"Original: {messy_data.tolist()}")

# This would error: messy_data.astype(int)

# Safe conversion
clean_data = pd.to_numeric(messy_data, errors='coerce')
print(f"After to_numeric: {clean_data.tolist()}")

In [ ]:
# pd.to_datetime() - convert strings to datetime
dates = pd.Series(['2023-01-15', '2023-02-20', '2023-03-25'])
print(f"Original type: {dates.dtype}")

dates_dt = pd.to_datetime(dates)
print(f"After to_datetime: {dates_dt.dtype}")
print(f"Values: {dates_dt.tolist()}")

## pd.Categorical: Efficient Categorical Data

In [ ]:
# Memory comparison: object vs categorical
df_copy = df.copy()

print(f"Sex memory as object: {df_copy['Sex'].memory_usage(deep=True):,} bytes")

df_copy['Sex_cat'] = df_copy['Sex'].astype('category')
print(f"Sex memory as category: {df_copy['Sex_cat'].memory_usage(deep=True):,} bytes")

In [ ]:
# Categorical with ordering
df_copy['Pclass_cat'] = pd.Categorical(
    df_copy['Pclass'],
    categories=[1, 2, 3],
    ordered=True
)

print(f"Categories: {df_copy['Pclass_cat'].cat.categories.tolist()}")
print(f"Ordered: {df_copy['Pclass_cat'].cat.ordered}")

# Now comparisons work
print(f"\nPassengers in class better than 2: {(df_copy['Pclass_cat'] < 2).sum()}")

## String Operations with .str Accessor

In [ ]:
# .str accessor provides vectorized string operations
names = df['Name'].head()
print("Original names:")
print(names.tolist())

In [ ]:
# str.lower(), str.upper()
print("Lowercase:")
print(df['Name'].str.lower().head().tolist())

In [ ]:
# str.contains() - regex search
# Find passengers with 'Mrs' in their name
mrs_passengers = df[df['Name'].str.contains('Mrs', na=False)]
print(f"Passengers with 'Mrs': {len(mrs_passengers)}")

In [ ]:
# str.split() - split strings
# Names are in format: "Last, Title. First"
# Extract last names
df_copy = df.copy()
df_copy['LastName'] = df_copy['Name'].str.split(',').str[0]
print("Extracted last names:")
print(df_copy['LastName'].head())

In [ ]:
# str.extract() - regex extraction
# Extract titles (Mr, Mrs, Miss, etc.)
df_copy['Title'] = df_copy['Name'].str.extract(r', ([A-Za-z]+)\.')
print("Extracted titles:")
print(df_copy['Title'].value_counts())

In [ ]:
# Other useful .str methods
s = pd.Series(['  hello  ', 'WORLD', 'Test String'])

print(f"Original: {s.tolist()}")
print(f"strip(): {s.str.strip().tolist()}")
print(f"title(): {s.str.title().tolist()}")
print(f"len(): {s.str.len().tolist()}")
print(f"replace: {s.str.replace('l', 'L').tolist()}")

## Exercises

### Exercise 1: Missing Value Analysis

For the Titanic dataset:
1. Which columns have more than 50% missing values?
2. How many rows have NO missing values?
3. Create a heatmap of missing values

In [ ]:
# Analyze missing values
# YOUR CODE HERE

### Exercise 2: Imputation

Create a cleaned version of the dataset where:
1. Age is filled with the median age
2. Embarked is filled with the mode
3. The Cabin column is dropped (too many missing)

In [ ]:
# Clean the dataset
# YOUR CODE HERE

### Exercise 3: String Operations

Using the Name column:
1. Count how many passengers have 'Master' in their title
2. Extract the title (Mr, Mrs, Miss, etc.) into a new column
3. Find all passengers whose names start with 'A'

In [ ]:
# String operations practice
# YOUR CODE HERE

### Exercise 4: Data Type Optimization

Convert the following columns to more appropriate types and measure memory savings:
1. Sex → category
2. Embarked → category
3. Survived → bool

In [ ]:
# Optimize data types and measure savings
# YOUR CODE HERE